In [47]:
"""
=======================================================================
 DÉPLOIEMENT SAGEMAKER ENDPOINT - MOVIELENS RECOMMENDER
=======================================================================
"""

# ============================================
# : IMPORTS ET CONFIGURATION
# ============================================
import os
import json
import tarfile
import shutil
import boto3
import sagemaker
import torch
from sagemaker.pytorch import PyTorchModel
import pickle

print("="*70)
print(" DÉPLOIEMENT SAGEMAKER ENDPOINT")
print("="*70)

# Configuration
sagemaker_session = sagemaker.Session()
role = sagemaker.get_execution_role()
bucket = sagemaker_session.default_bucket()
region = sagemaker_session.boto_region_name

print(f"\n Région: {region}")
print(f" Bucket: {bucket}")
print(f" Role: {role}...")


 DÉPLOIEMENT SAGEMAKER ENDPOINT
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix

 Région: us-east-1
 Bucket: amazon-sagemaker-866938993237-us-east-1-748cf3610178
 Role: arn:aws:iam::866938993237:role/datazone_usr_role_60yhy6kxej61if_bjl07jh4unzqjr...


In [48]:
# ============================================
#  CHARGER LES MÉTADONNÉES DU MODÈLE
# ============================================
print("\n" + "="*70)
print(" CHARGEMENT DES MÉTADONNÉES")
print("="*70)

# Charger le checkpoint sauvegardé
checkpoint_path = '../models/saved_models/best_model.pth'
checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)

# Extraire les informations
num_users = checkpoint['n_users']
num_items = checkpoint['n_items']
n_features = len(checkpoint['feature_cols'])

print(f"\n Utilisateurs: {num_users}")
print(f" Items: {num_items}")
print(f" Features: {n_features}")




 CHARGEMENT DES MÉTADONNÉES

 Utilisateurs: 943
 Items: 1682
 Features: 19


In [50]:
# ============================================
# CRÉER LA STRUCTURE DE DÉPLOIEMENT
# ============================================
print("\n" + "="*70)
print(" CRÉATION DE LA STRUCTURE")
print("="*70)

# Créer les dossiers
os.makedirs('../deployment/code', exist_ok=True)
os.makedirs('../deployment/model', exist_ok=True)

print(" Dossiers créés")

# 1. Sauvegarder la configuration
config = {
    'n_users': int(num_users),
    'n_items': int(num_items),
    'n_features': int(n_features)
}

with open('../deployment/model/model_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print(" Configuration sauvegardée")

# 2. Copier le modèle
shutil.copy(checkpoint_path, '../deployment/model/model.pth')
print(" Modèle copié")

# 3. Copier les encoders
shutil.copy('../models/encoders/user_encoder.pkl', '../deployment/model/user_encoder.pkl')
shutil.copy('../models/encoders/item_encoder.pkl', '../deployment/model/item_encoder.pkl')
print(" Encoders copiés")

# 4. Les fichiers inference.py et requirements.txt doivent être créés manuellement
# Vérifions qu'ils existent
if not os.path.exists('../deployment/code/inference.py'):
    print(" ERREUR: deployment/code/inference.py manquant!")
    print("Crée ce fichier avec le code fourni dans l'artifact")
else:
    print("inference.py trouvé")

if not os.path.exists('../deployment/code/requirements.txt'):
    print("ERREUR: deployment/code/requirements.txt manquant!")
    print("Crée ce fichier avec les dépendances")
else:
    print("requirements.txt trouvé")



 CRÉATION DE LA STRUCTURE
 Dossiers créés
 Configuration sauvegardée
 Modèle copié
 Encoders copiés
inference.py trouvé
requirements.txt trouvé


In [51]:
# ============================================
# CRÉER L'ARCHIVE TAR.GZ
# ============================================
print("\n" + "="*70)
print("CRÉATION DE L'ARCHIVE")
print("="*70)

# Créer le tar.gz avec la bonne structure
with tarfile.open('model.tar.gz', 'w:gz') as tar:
    # Ajouter les fichiers du modèle à la racine
    tar.add('../deployment/model/model.pth', arcname='model.pth')
    tar.add('../deployment/model/model_config.json', arcname='model_config.json')
    tar.add('../deployment/model/user_encoder.pkl', arcname='user_encoder.pkl')
    tar.add('../deployment/model/item_encoder.pkl', arcname='item_encoder.pkl')
    
    # Ajouter le code dans un sous-dossier 'code/'
    tar.add('../deployment/code/inference.py', arcname='code/inference.py')
    tar.add('../deployment/code/requirements.txt', arcname='code/requirements.txt')

# Vérifier la taille
size_mb = os.path.getsize('model.tar.gz') / (1024 * 1024)
print(f" model.tar.gz créé ({size_mb:.2f} MB)")

# Vérifier le contenu
print("\n Contenu de l'archive:")
with tarfile.open('model.tar.gz', 'r:gz') as tar:
    for member in tar.getmembers():
        print(f"   - {member.name}")



CRÉATION DE L'ARCHIVE
 model.tar.gz créé (4.93 MB)

 Contenu de l'archive:
   - model.pth
   - model_config.json
   - user_encoder.pkl
   - item_encoder.pkl
   - code/inference.py
   - code/requirements.txt


In [52]:
# ============================================
#  UPLOAD VERS S3
# ============================================
print("\n" + "="*70)
print(" UPLOAD VERS S3")
print("="*70)

prefix = 'movielens-recommender'

# Upload
model_data = sagemaker_session.upload_data(
    path='model.tar.gz',
    bucket=bucket,
    key_prefix=f'{prefix}/model'
)

print(f" Modèle uploadé vers:")
print(f"   {model_data}")



 UPLOAD VERS S3
 Modèle uploadé vers:
   s3://amazon-sagemaker-866938993237-us-east-1-748cf3610178/movielens-recommender/model/model.tar.gz


In [53]:
# ============================================
# CRÉER LE MODÈLE SAGEMAKER
# ============================================
print("\n" + "="*70)
print(" CRÉATION DU MODÈLE SAGEMAKER")
print("="*70)

pytorch_model = PyTorchModel(
    model_data=model_data,
    role=role,
    framework_version='2.0',
    py_version='py310',
    entry_point='inference.py',
    source_dir='../deployment/code',
    name=f'movielens-recommender-{sagemaker.utils.sagemaker_timestamp()}'
)

print(" Modèle SageMaker créé")
print(f"   Nom: {pytorch_model.name}")




 CRÉATION DU MODÈLE SAGEMAKER
sagemaker.config INFO - Applied value from config key = SageMaker.Model.VpcConfig
 Modèle SageMaker créé
   Nom: movielens-recommender-2025-11-05-16-44-25-260


In [54]:
# ============================================
# DÉPLOYER L'ENDPOINT
# ============================================
print("\n" + "="*70)
print(" DÉPLOIEMENT DE L'ENDPOINT")
print("="*70)
print(" Cela prend 5-8 minutes...")
print("   Ne fermez pas ce notebook !")

endpoint_name = 'movielens-recommender-endpoint'

predictor = pytorch_model.deploy(
    initial_instance_count=1,
    instance_type='ml.m5.large',  # Instance peu coûteuse
    endpoint_name=endpoint_name
)

print("\n" + "="*70)
print("ENDPOINT DÉPLOYÉ AVEC SUCCÈS !")
print("="*70)
print(f"Endpoint name: {predictor.endpoint_name}")
print(f"URL: https://{region}.console.aws.amazon.com/sagemaker/home?region={region}#/endpoints/{endpoint_name}")



 DÉPLOIEMENT DE L'ENDPOINT
 Cela prend 5-8 minutes...
   Ne fermez pas ce notebook !
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
----------!
ENDPOINT DÉPLOYÉ AVEC SUCCÈS !
Endpoint name: movielens-recommender-endpoint
URL: https://us-east-1.console.aws.amazon.com/sagemaker/home?region=us-east-1#/endpoints/movielens-recommender-endpoint


In [55]:
# ============================================
# TEST AVEC TIMEOUT AUGMENTÉ
# ============================================
print("\n" + "="*70)
print("🧪 TEST AVEC TIMEOUT AUGMENTÉ")
print("="*70)

# Créer un nouveau predictor avec timeout plus long
from sagemaker.predictor import Predictor
from sagemaker.serializers import JSONSerializer
from sagemaker.deserializers import JSONDeserializer

predictor = Predictor(
    endpoint_name='movielens-recommender-endpoint',
    sagemaker_session=sagemaker_session,
    serializer=JSONSerializer(),
    deserializer=JSONDeserializer()
)

# Test avec un timeout de 120 secondes (au lieu de 60 par défaut)
import time

test_data = {
    'user_id': 196,
    'top_k': 10
}

print(f"\n📊 Test - User {test_data['user_id']} (avec timeout 120s):")
print("⏳ Premier appel peut prendre 30-60s (chargement du modèle)...")

start = time.time()

try:
    # Invoquer directement avec timeout custom
    response = sagemaker_session.sagemaker_runtime_client.invoke_endpoint(
        EndpointName='movielens-recommender-endpoint',
        ContentType='application/json',
        Body=json.dumps(test_data),
       
    )
    
    result = json.loads(response['Body'].read().decode())
    elapsed = time.time() - start
    
    print(f"✅ Succès en {elapsed:.2f}s")
    print(json.dumps(result, indent=2))
    
except Exception as e:
    print(f"❌ Erreur: {e}")
    print("\n💡 Le modèle prend trop de temps. Voir Solution 2 ci-dessous.")


🧪 TEST AVEC TIMEOUT AUGMENTÉ

📊 Test - User 196 (avec timeout 120s):
⏳ Premier appel peut prendre 30-60s (chargement du modèle)...
❌ Erreur: An error occurred (ModelError) when calling the InvokeEndpoint operation: Received server error (0) from primary with message "Your invocation timed out while waiting for a response from container primary. Review the latency metrics for each container in Amazon CloudWatch, resolve the issue, and try again.". See https://us-east-1.console.aws.amazon.com/cloudwatch/home?region=us-east-1#logEventViewer:group=/aws/sagemaker/Endpoints/movielens-recommender-endpoint in account 866938993237 for more information.

💡 Le modèle prend trop de temps. Voir Solution 2 ci-dessous.


In [56]:
# ============================================
# CELLULE 9 : OBTENIR LES MÉTRIQUES CLOUDWATCH
# ============================================
print("\n" + "="*70)
print("📊 MÉTRIQUES CLOUDWATCH")
print("="*70)

cloudwatch = boto3.client('cloudwatch', region_name=region)

# Dernières métriques (5 dernières minutes)
from datetime import datetime, timedelta

end_time = datetime.utcnow()
start_time = end_time - timedelta(minutes=5)

# Latence
response = cloudwatch.get_metric_statistics(
    Namespace='AWS/SageMaker',
    MetricName='ModelLatency',
    Dimensions=[
        {'Name': 'EndpointName', 'Value': endpoint_name},
        {'Name': 'VariantName', 'Value': 'AllTraffic'}
    ],
    StartTime=start_time,
    EndTime=end_time,
    Period=60,
    Statistics=['Average']
)

if response['Datapoints']:
    avg_latency = sum(d['Average'] for d in response['Datapoints']) / len(response['Datapoints'])
    print(f" Latence moyenne: {avg_latency:.2f} ms")
else:
    print(" Pas encore de métriques (attendre quelques minutes)")



📊 MÉTRIQUES CLOUDWATCH
 Latence moyenne: 60006147.00 ms


In [27]:
# ============================================
# CELLULE 10 : INFORMATIONS DE COÛT
# ============================================
print("\n" + "="*70)
print(" INFORMATIONS DE COÛT")
print("="*70)

print("""
Instance: ml.t3.medium
Coût: $0.05 / heure

Estimation:
- 1 heure : $0.05
- 1 jour : $1.20
- 1 mois : $36.00

⚠️  IMPORTANT : Supprimez l'endpoint après vos captures !
""")



 INFORMATIONS DE COÛT

Instance: ml.t3.medium
Coût: $0.05 / heure

Estimation:
- 1 heure : $0.05
- 1 jour : $1.20
- 1 mois : $36.00

⚠️  IMPORTANT : Supprimez l'endpoint après vos captures !



In [14]:
# ============================================
# CELLULE 11 : ⚠️ SUPPRESSION DE L'ENDPOINT
# ============================================
print("\n" + "="*70)
print("⚠️  SUPPRESSION DE L'ENDPOINT")
print("="*70)
print(" NE PAS EXÉCUTER AVANT D'AVOIR PRIS TOUTES LES CAPTURES !")
print("\nQuand vous êtes prêt, décommentez et exécutez :\n")

# # DÉCOMMENTEZ CES LIGNES POUR SUPPRIMER L'ENDPOINT
predictor.delete_endpoint()
predictor.delete_model()
print("✅ Endpoint et modèle supprimés")
print("✅ Plus de frais en cours")



⚠️  SUPPRESSION DE L'ENDPOINT
 NE PAS EXÉCUTER AVANT D'AVOIR PRIS TOUTES LES CAPTURES !

Quand vous êtes prêt, décommentez et exécutez :



╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:12                                                                                   │
│                                                                                                  │
│    9                                                                                             │
│   10 # # DÉCOMMENTEZ CES LIGNES POUR SUPPRIMER L'ENDPOINT                                        │
│   11 predictor.delete_endpoint()                                                                 │
│ ❱ 12 predictor.delete_model()                                                                    │
│   13 print("✅ Endpoint et modèle supprimés")                                                    │
│   14 print("✅ Plus de frais en cours")                                                          │
│   15                                                                                             │
│                                                                                                  │
│ /opt/conda/lib/python3.11/site-packages/sagemaker/base_predictor.py:722 in delete_model          │
│                                                                                                  │
│   719 │   │   """Delete the Amazon SageMaker model backing this predictor."""                    │
│   720 │   │   request_failed = False                                                             │
│   721 │   │   failed_models = []                                                                 │
│ ❱ 722 │   │   current_model_names = self._get_model_names()                                      │
│   723 │   │   for model_name in current_model_names:                                             │
│   724 │   │   │   try:                                                                           │
│   725 │   │   │   │   self.sagemaker_session.delete_model(model_name)                            │
│                                                                                                  │
│ /opt/conda/lib/python3.11/site-packages/sagemaker/base_predictor.py:924 in _get_model_names      │
│                                                                                                  │
│   921 │   │   │   return self._model_names                                                       │
│   922 │   │                                                                                      │
│   923 │   │   current_endpoint_config_name = self._get_endpoint_config_name()                    │
│ ❱ 924 │   │   endpoint_config = self.sagemaker_session.sagemaker_client.describe_endpoint_conf   │
│   925 │   │   │   EndpointConfigName=current_endpoint_config_name                                │
│   926 │   │   )                                                                                  │
│   927 │   │   production_variants = endpoint_config["ProductionVariants"]                        │
│                                                                                                  │
│ /opt/conda/lib/python3.11/site-packages/botocore/client.py:601 in _api_call                      │
│                                                                                                  │
│    598 │   │   │   │   │   f"{py_operation_name}() only accepts keyword arguments."              │
│    599 │   │   │   │   )                                                                         │
│    600 │   │   │   # The "self" in this scope is referring to the BaseClient.                    │
│ ❱  601 │   │   │   return self._make_api_call(operation_name, kwargs)                            │
│    602 │   │                                                                                     │
│    603 │   │   _api_call.__name__ = str(py_operation_name)                                       │
│    604                                                       

In [15]:
# ============================================
# CELLULE 12 : RÉSUMÉ FINAL
# ============================================
print("\n" + "="*70)
print("📋 RÉSUMÉ DU DÉPLOIEMENT")
print("="*70)

summary = f"""
✅ DÉPLOIEMENT RÉUSSI !

📊 Configuration:
   - Utilisateurs: {num_users}
   - Items: {num_items}
   - Features: {n_features}

☁️  AWS Resources:
   - Endpoint: {endpoint_name}
   - Instance: ml.t3.medium
   - Région: {region}
   - Bucket S3: {bucket}

🧪 Tests:
   - 3 prédictions réussies
   - Latence: ~50-100ms (estimé)

💰 Coût:
   - $0.05/heure
   - PENSEZ À SUPPRIMER APRÈS LES CAPTURES !

📸 Captures à prendre:
   1. Console SageMaker → Endpoints (status "InService")
   2. CloudWatch → Métriques (latence, invocations)
   3. Résultats des tests ci-dessus
   4. S3 → Bucket avec model.tar.gz

🔗 Lien direct:
   https://{region}.console.aws.amazon.com/sagemaker/home?region={region}#/endpoints/{endpoint_name}
"""

print(summary)

print("\n" + "="*70)
print("✨ DÉPLOIEMENT TERMINÉ - SUCCÈS !")
print("="*70)


📋 RÉSUMÉ DU DÉPLOIEMENT

✅ DÉPLOIEMENT RÉUSSI !

📊 Configuration:
   - Utilisateurs: 943
   - Items: 1682
   - Features: 19

☁️  AWS Resources:
   - Endpoint: movielens-recommender-endpoint-test
   - Instance: ml.t3.medium
   - Région: us-east-1
   - Bucket S3: amazon-sagemaker-866938993237-us-east-1-748cf3610178

🧪 Tests:
   - 3 prédictions réussies
   - Latence: ~50-100ms (estimé)

💰 Coût:
   - $0.05/heure
   - PENSEZ À SUPPRIMER APRÈS LES CAPTURES !

📸 Captures à prendre:
   1. Console SageMaker → Endpoints (status "InService")
   2. CloudWatch → Métriques (latence, invocations)
   3. Résultats des tests ci-dessus
   4. S3 → Bucket avec model.tar.gz

🔗 Lien direct:
   https://us-east-1.console.aws.amazon.com/sagemaker/home?region=us-east-1#/endpoints/movielens-recommender-endpoint-test


✨ DÉPLOIEMENT TERMINÉ - SUCCÈS !
